# Metadata Filtering을 이용한 검색 범위 제어

Metadata Filtering(메타데이터 필터링)은 벡터 유사도를 계산하기 전에 **구조화 조건을 만족하는 문서로 검색 후보를 제한**하는 방식이다. 질의와 의미가 가까워도 저자·카테고리·작성일 조건이 맞지 않으면 결과에서 제외한다.

### 핵심 용어

- `metadata`: 문서 본문을 설명하는 `doc_id`, `author`, `category` 같은 key-value 정보이다.
- `filter`: metadata에 적용할 검색 조건을 Python 딕셔너리로 표현한 값이다.
- 검색 후보: 필터를 통과해 벡터 유사도 순위를 계산할 문서 집합이다.
- `Document`: LangChain이 본문 `page_content`와 부가정보 `metadata`를 함께 담는 객체이다.
- `list[Document]`: 검색된 여러 Document가 유사도 순서로 담긴 목록이다.
- `$in`: metadata 값이 주어진 여러 값 중 하나에 해당하는지 검사하는 Pinecone 필터 연산자이다.
- Upsert: 같은 ID가 없으면 새로 추가하고, 있으면 기존 레코드를 갱신하는 저장 작업이다.
- Self-query: LLM이 자연어 질문에서 검색어와 metadata 조건을 분리하는 방식이다.

RRF(Reciprocal Rank Fusion)가 여러 검색 순위를 결합한다면, Metadata Filtering은 순위를 계산할 후보의 범위를 먼저 제한한다. 이 노트북은 `adv-rag-meta` 인덱스에 본문과 metadata를 적재하고, `$in` 조건을 통과한 `list[Document]`를 확인한다.


## Metadata Filtering 실행 패키지 준비

`%pip`은 현재 Jupyter 커널에 패키지를 설치하는 명령이다. 설치 후 기존 모듈이 계속 로드되면 커널을 한 번 재시작한다.

- `pandas`: CSV를 DataFrame으로 읽고 행·열을 처리한다.
- `langchain`: LangChain의 핵심 자료형과 실행 인터페이스를 제공한다.
- `langchain-openai`: 문서와 질의를 OpenAI 임베딩 벡터로 변환한다.
- `langchain-pinecone`: LangChain `Document`와 Pinecone 저장·검색을 연결한다.
- `pinecone`: 원격 인덱스를 생성하고 상태를 확인한다.
- `python-dotenv`: `.env`의 API key와 설정을 환경 변수로 읽는다.
- `gdown`: Google Drive의 공개 실습 데이터를 다운로드한다.


In [1]:
%pip install -U pandas langchain langchain-openai langchain-pinecone pinecone python-dotenv gdown

  Using cached pinecone-9.1.0-cp310-abi3-win_amd64.whl.metadata (6.3 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## `.env`와 Pinecone 설정 준비

`.env`에 저장한 OpenAI와 Pinecone API key를 환경 변수로 불러온다. API key는 출력하지 않고 뒤의 SDK 객체가 자동으로 사용한다.

- `find_dotenv()`: 현재 작업 위치에서 `.env` 파일을 찾는다.
- `load_dotenv()`: 찾은 파일의 key-value를 `os.environ`에 추가한다.
- `os.getenv()`: 환경 변수를 읽고, 값이 없으면 두 번째 인자의 기본값을 사용한다.
- `index`: 같은 vector 길이와 거리 계산 방식을 사용하는 Pinecone 레코드의 저장 단위이다.
- `namespace`: 하나의 index 안에서 레코드를 논리적으로 나누는 공간이다. 이 실습은 값을 지정하지 않아 default namespace를 사용한다.

인덱스 이름·cloud·region·dimension·metric은 인덱스 생성, 임베딩, 검색 단계에서 같은 값을 사용해야 한다.


In [2]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(dotenv_path=find_dotenv(usecwd=True), override=False)

PINECONE_META_INDEX_NAME = 'adv-rag-meta'
PINECONE_INDEX_REGION = os.getenv('PINECONE_INDEX_REGION', 'us-east-1')
PINECONE_INDEX_CLOUD = os.getenv('PINECONE_INDEX_CLOUD', 'aws')
PINECONE_INDEX_METRIC = os.getenv('PINECONE_INDEX_METRIC', 'cosine')
PINECONE_INDEX_DIMENSION = int(os.getenv('PINECONE_INDEX_DIMENSION', '1536'))
OPENAI_EMBEDDING_MODEL = os.getenv(
    'OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'
)


## 메타데이터 포함 문서 다운로드

`gdown`은 Google Drive에 공개된 파일을 파일 ID로 다운로드하는 도구이다. Jupyter의 `!`는 뒤에 이어진 명령을 Python이 아닌 시스템 셸에서 실행한다.

`documents_meta.csv`의 한 행은 하나의 검색 문서이다.

- `doc_id`: 문서를 구분하고 Pinecone ID로 사용할 고유 문자열이다.
- `content`: 임베딩하고 의미 유사도를 비교할 본문이다.
- `author`: 저자 필터에 사용할 단일 문자열이다.
- `category`: `역사;교육`처럼 세미콜론으로 연결된 카테고리 문자열이다. 뒤에서 `['역사', '교육']`로 변환한다.


In [3]:
!gdown 1pSvb5yHflp2nR5c-90dIUa7MjPjZNRb- -O documents_meta.csv

Downloading...
From: https://drive.google.com/uc?id=1pSvb5yHflp2nR5c-90dIUa7MjPjZNRb-
To: C:\SKN_AI\09_llm\07_advanced_rag\01_retrieval_optimization\documents_meta.csv

  0%|          | 0.00/15.9k [00:00<?, ?B/s]
100%|██████████| 15.9k/15.9k [00:00<00:00, 6.35MB/s]


## CSV를 DataFrame으로 읽기

`DataFrame`은 Pandas가 행과 열로 구성된 표 데이터를 담는 객체이다. `pd.read_csv()`는 CSV 파일을 읽어 DataFrame을 반환한다.

출력 표에서 다음을 확인한다.

- 행: D1부터 D30까지의 검색 문서 30개이다.
- `content`: 뒤에서 `Document.page_content`로 사용한다.
- `doc_id`·`author`·`category`: 뒤에서 `Document.metadata`로 사용한다.


In [4]:
import pandas as pd

documents_df = pd.read_csv("documents_meta.csv")
documents_df.head(3)

,doc_id,title,content,author,category
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(...",김민수,여행
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기...",이영희,음식
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet...",박지훈,문화;음악


## 메타데이터 전용 Pinecone 인덱스 준비

Pinecone은 임베딩 vector와 metadata를 저장·검색하는 관리형 Vector DB이다.

- `adv-rag`: 앞선 Dense Retrieval·RRF 실습에서 사용한 기존 검색 인덱스이다.
- `adv-rag-meta`: `author`·`category` metadata를 적재하고 다음 `05_self_query_retriever.ipynb`에서 재사용하기 위한 별도 인덱스이다.
- `list_indexes().names()`: 현재 Pinecone 프로젝트의 인덱스 이름 목록을 반환한다.
- `create_index()`: 지정한 이름과 vector 설정으로 인덱스를 생성한다.
- `dimension`: 저장할 임베딩 vector 하나의 길이이다.
- `metric`: vector 사이의 거리를 계산하는 방식이다.
- `ServerlessSpec`: 인덱스를 배치할 `cloud`와 `region`을 묶어 전달한다.
- `describe_index()`: 인덱스 설정과 상태를 반환한다. `status['ready']=True`면 문서를 저장할 수 있다.

`adv-rag-meta`가 이름 목록에 없을 때만 생성한 뒤 Ready 상태까지 대기한다. 이 경로는 `langchain-pinecone`과 함께 설치되는 호환 Pinecone SDK 버전을 기준으로 한다.


In [6]:
from time import sleep
from pinecone import Pinecone, ServerlessSpec

# Pinecone 연결 클라이언트 객체 (환경 변수 API 자동 인식)
pc = Pinecone()

# 같은 이름의 index가 존재하는지 확인하기 위해서
# index 이름 목록 조회
existing_index_names: list[str] = pc.list_indexes().names()

# 'adv-rag-meta'
if PINECONE_META_INDEX_NAME not in existing_index_names:
    pc.create_index(
        name=PINECONE_META_INDEX_NAME, # 'adv-rag-meta'
        dimension=PINECONE_INDEX_DIMENSION, # 1536
        metric=PINECONE_INDEX_METRIC, # 'cosine'
        spec=ServerlessSpec(
            cloud=PINECONE_INDEX_CLOUD, # 'aws'
            region=PINECONE_INDEX_REGION # 'us-east-1'
        ),
        timeout=-1 # 생성 요청만 하고 대기 없이 바로 다음 넘어감
    )


# 'adv-rag-meta' index가 준비 되었는지 확인
while not pc.describe_index(
        PINECONE_META_INDEX_NAME).status['ready']:
    print("아직 준비되지 않음...")
    sleep(1)

print(f"{PINECONE_META_INDEX_NAME} 인덱스 준비 완료!")

adv-rag-meta 인덱스 준비 완료!


## OpenAI 임베딩과 Pinecone Vector Store 연결

`OpenAIEmbeddings`는 문서와 질의 문자열을 의미를 표현하는 숫자 vector로 변환한다. `PineconeVectorStore`는 LangChain에서 Pinecone의 문서 적재와 유사도 검색을 같은 인터페이스로 사용하게 하는 연결 객체이다.

- `model`: 문자열을 vector로 변환할 OpenAI 임베딩 모델이다.
- `dimensions`: 모델이 출력할 vector 길이이며 Pinecone index의 `dimension`과 같아야 한다.
- `index_name`: 저장과 검색 대상인 `adv-rag-meta`를 선택한다.
- `embedding`: 문서 적재와 query 검색 모두에 같은 임베딩 객체를 사용한다.
- `namespace`: 인자를 생략해 default namespace를 사용한다. 다음 Self-query도 같은 index와 default namespace에 연결해야 한다.

이 객체를 만들면 index 연결 정보만 준비된다. OpenAI 임베딩 비용과 문서 upsert는 뒤의 `add_documents()`를 실행할 때 발생한다.


In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    dimensions=PINECONE_INDEX_DIMENSION
)

vector_store = PineconeVectorStore(
    index_name=PINECONE_META_INDEX_NAME,
    embedding=embeddings
)

## DataFrame 행을 metadata가 있는 Document로 변환

`Document`는 LangChain의 검색 문서 자료형이다. 본문과 필터 정보를 한 객체에 결합한다.

- `page_content`: 임베딩하고 query와 의미 유사도를 비교할 본문 문자열이다.
- `metadata`: 필터, 식별, 결과 표시에 사용할 부가정보 딕셔너리이다.
- `iterrows()`: DataFrame을 한 행씩 순회하며 `(행 index, 행 Series)`를 반환한다.
- `Series`: DataFrame의 한 행 또는 한 열을 key-value 형태로 담는 Pandas 객체이다.
- `split(';')`: `'역사;교육'` 문자열을 `['역사', '교육']` 목록으로 변환한다.

`author`는 단일 문자열로 유지하고 `category`는 여러 필터 값을 가질 수 있는 문자열 목록으로 바꾼다. 변환한 `docs`는 다음 upsert 셀의 입력이 된다.


In [10]:
from langchain_core.documents import Document

docs: list[Document] = []

for _, row in documents_df.iterrows():
    metadata = {
        'doc_id': str(row['doc_id']),
        'author': str(row['author']),

        # category: 세미콜론 문자열을 `$in` 필터에 사용할 list[str]로 변환
        'category': str(row['category']).split(';'),
    }

    document = Document(
        page_content=str(row['content']),
        metadata=metadata,
    )

    docs.append(document)

print(type(docs[0]))

<class 'langchain_core.documents.base.Document'>


## Document vector와 metadata를 Pinecone에 upsert

Upsert는 ID가 없으면 새 레코드를 추가하고, 같은 ID가 있으면 기존 레코드를 갱신하는 작업이다. 고정 ID를 사용하면 셀을 다시 실행해도 동일 문서가 중복 누적되지 않는다.

- `doc_ids`: `docs`의 각 `metadata['doc_id']`를 같은 순서로 모은 `list[str]`이다.
- `add_documents()`: `page_content`를 임베딩한 뒤 vector와 metadata를 Pinecone에 저장한다.
- `documents`: 저장할 `Document` 목록이다.
- `ids`: 각 `Document`와 같은 위치에 대응하는 Pinecone 레코드 ID 목록이다.
- 반환값: 실제로 저장한 ID의 `list[str]`이다.

이 단계는 OpenAI Embeddings 호출과 Pinecone 쓰기 요청을 발생시킨다. 완료하면 다음 `05_self_query_retriever.ipynb`가 같은 index와 default namespace를 재사용한다.


In [12]:
# 문서 id 목록
doc_ids: list[str] = [
    doc.metadata['doc_id'] for doc in docs
]

# Pinecone Index에 Document 추가 후 id 목록 반환 받기
stored_ids: list[str] = vector_store.add_documents(
    documents=docs,
    ids=doc_ids
)

print(len(stored_ids))
print(stored_ids[0])


30
D1


## `$in`으로 카테고리를 제한한 유사도 검색

`similarity_search()`는 query를 vector로 바꾸고 문서 vector와 의미가 가까운 문서를 찾는다. `filter`를 함께 전달하면 Pinecone은 먼저 metadata 조건을 통과한 후보만 남기고, 그 안에서 vector 유사도 순위를 계산한다.

- `query`: 의미가 가까운 문서를 찾을 자연어 질의이다.
- `k`: 필터를 통과한 후보 중 최대로 반환할 `Document` 개수이다.
- `filter`: metadata 필드, 연산자, 비교 값으로 구성된 조건 딕셔너리이다.
- `category`: 검사할 metadata 필드 이름이다.
- `$in`: 문서의 `category` 목록에 지정 값 중 하나라도 포함되면 통과시키는 Pinecone 필터 연산자이다.
- 반환값: 조건을 통과한 문서가 유사도 순서로 배치된 `list[Document]`이다.

예제의 `{'category': {'$in': ['역사', '기술']}}`는 `category`에 `역사` 또는 `기술`이 있는 문서만 허용한다. 문법과 연산자는 [Pinecone Metadata Filtering 공식 문서](https://docs.pinecone.io/guides/search/filter-by-metadata)에서 확인할 수 있다.


In [16]:
query = '훈민정음' # 유사도 검색을 위한 질의어

# metadata - category 필터링 준비
category_filter = {
    'category': {
        '$in': ['역사', '기술']
    }
}

# 필터링이 적용된 유사도 검색
results: list[Document] = vector_store.similarity_search(
    query=query,
    k=5,

    # 유사도 순위 계산 전에
    # 조회된 Document의 metadata중
    # 'category' 값을 얻어와 '$in' 필터를 이용해서
    #  ['역사', '기술'] 중 하나라도 존재하면 통과, 둘다 없으면 필터링
    filter=category_filter
)

for doc in results:
    print(f'{doc.metadata["doc_id"]} / {doc.metadata["author"]}:')
    print(doc.metadata['category'])
    print(doc.page_content)
    print()


D4 / 최수정:
['역사', '교육']
세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입니다. 그가 훈민정음을 만든 배경에는 백성들의 문맹 문제 해결과 국가 통치 효율화가 있었습니다. 세종대왕의 업적은 한국 문화와 문자 체계에 지대한 영향을 미쳤으며, 훈민정음 해례본은 유네스코 세계기록유산으로 등재되었습니다.

D30 / 윤아름:
['기술', 'AI', '검색']
Multi-Hop Retrieval은 한 단계의 검색으로 해결되지 않는 복합 질문에 대응하기 위해, 여러 단계(홉)로 나눠서 검색을 수행하는 기법입니다. 예를 들어, “세종대왕이 훈민정음을 만든 이유를 바탕으로 AI 윤리 가이드라인 사례를 찾고 싶다”는 질문에서, 1단계로 ‘세종대왕 훈민정음 배경’(→D4 자료), 2단계로 ‘AI 윤리 가이드라인 사례’(→D7 또는 D25)로 연결해 답을 도출합니다. 단계별 결과를 종합해 최종 순위를 매깁니다.

D15 / 정우성:
['문화', '음악', '역사']
가야금은 한국 전통 현악기로, 12개의 줄을 가진 칠현금(七絃琴) 계열 악기입니다. 역사적으로 가야 시대에 기원하며, 고려 말·조선 초기에 궁중 음악으로 발전했습니다. 현대에는 거문고·해금 등과 함께 국악 앙상블에서 주로 사용되며, 연주법은 손끝으로 줄을 튕기는 방식입니다.

D18 / 오정연:
['기술', '소프트웨어', '설치']
윈도우즈 환경에서 한글이 깨지지 않고 시각화(그래프·차트)할 때는, 한글 폰트를 시스템에 설치한 뒤 matplotlib에 폰트 경로를 지정해야 합니다. 예를 들어, “NanumGothic” 또는 “Malgun Gothic” 폰트를 설치한 후, 파이썬 코드에 `plt.rc("font", family="Malgun Gothic")`를 추가합니다. 이 과정을 통해 차트 제목·축 레이블·범례 등에서 한글이 올바르게 출력됩니다.

D9 / 신동엽:
['문화', '음악', '역사']
판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작

## 다음 단계: 개발자 필터에서 Self-query로

이 실습에서는 개발자가 `filter={'category': {'$in': [...]}}` 딕셔너리를 직접 작성했다. 조건이 명확하고 고정되어 있을 때 사용하기 좋지만, 사용자의 자연어 문장을 개발자가 매번 필터 딕셔너리로 바꾸어야 한다.

- Self-query: LLM이 자연어 질문에서 검색어와 metadata 조건을 분리하는 방식이다.
- `StructuredQuery`: 검색어와 구조화된 필터 조건을 함께 담는 결과 객체이다.
- 다음 실습: `05_self_query_retriever.ipynb`가 같은 `adv-rag-meta`의 default namespace를 연결하고 사용자 조건을 Pinecone filter로 변환한다.
